In [1]:
from config import MODEL_CONFIGS
from pathlib import Path
import pandas as pd
from datasets import Dataset
import json
from transformers import AutoTokenizer
from patching import create_diff_patch, remove_comments, remove_file_headers, apply_generated_patch, get_similarity

WORK_DIR = Path.cwd()
DS_TEST_PATH = WORK_DIR / "dataset" / "split" / "test_dataset.jsonl"
ds_test = Dataset.from_pandas(pd.read_json(DS_TEST_PATH, lines=True))

In [4]:
from tqdm import tqdm

failed = []
sum_sim = 0.0
count = 0
sims = []
failed_applying = []

pbar = tqdm(ds_test, desc="Evaluating", unit="example")

for example in pbar:
    id = example["id"]
    bad_code = example["bad_code"]
    good_code = example["good_code"]

    patch = create_diff_patch(bad_code, good_code)
    fixed_code, st = apply_generated_patch(bad_code, patch)
    if not st:
        failed_applying.append(st)

    sim = get_similarity(good_code, fixed_code)

    count += 1
    sum_sim += sim
    avg_sim = sum_sim / count
    sims.append(sim)

    if sim != 1.0:
        failed.append(id)

    pbar.set_postfix(
        last_sim=f"{sim:.4f}",
        avg_sim=f"{avg_sim:.4f}"
    )

passed = len(ds_test) - len(failed)

print(f"Passed: {passed}, Failed: {len(failed)}")
print(f"Average similarity: {avg_sim:.4f}")


Evaluating: 100%|██████████| 1172/1172 [02:11<00:00,  8.92example/s, avg_sim=0.9992, last_sim=0.9998]

Passed: 203, Failed: 969
Average similarity: 0.9992


In [3]:
print(len(failed_applying))

248


In [ ]:
idx = sims.index(min(sims))
example = ds_test[idx]

print(example["diff_patch"])

In [ ]:
print(example["bad_code"])

In [ ]:
fixed_code = apply_generated_patch(example["bad_code"], example["diff_patch"])
print(fixed_code)

In [ ]:
import matplotlib.pyplot as plt
plt.plot(sims)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
plt.plot(sims)
plt.show()

In [ ]:
for example in ds_test:
    id = example["id"]
    diff_patch 

In [ ]:
print(ds_test[0]["response"])

In [ ]:
from diff_match_patch import diff_match_patch

dmp = diff_match_patch()
patches = dmp.patch_make(bc, gc)
patch_text = dmp.patch_toText(patches)

# apply the patch
new_text, results = dmp.patch_apply(patches, bc)

In [ ]:
print(patch_text)

In [ ]:
import subprocess
import os

DIFF = r"C:\Program Files\Git\usr\bin\diff.exe"
PATCH = r"C:\Program Files\Git\usr\bin\patch.exe"

OLD = "__old.txt"
NEW = "__new.txt"
TARGET = OLD


def create_diff_patch(old_code: str, new_code: str) -> str:
    """Creates a diff patch between old and new code, using DIFF"""
    
    try:
        with open(OLD, "w", newline="\n") as f:
            f.write(old_code if old_code.endswith("\n") else old_code + "\n")

        with open(NEW, "w", newline="\n") as f:
            f.write(new_code if new_code.endswith("\n") else new_code + "\n")

        result = subprocess.run(
            [DIFF, "-u", OLD, NEW],
            text=True,
            capture_output=True,
        )

        if result.returncode not in (0, 1):
            raise RuntimeError(result.stderr)

        return result.stdout
    
    except Exception:
        raise

    finally:
        OLD.unlink(missing_ok=True)
        NEW.unlink(missing_ok=True)
        

def apply_diff_patch(code: str, patch: str) -> tuple[str, bool]:
    """Applies a diff patch to given code through PATCH"""

    try:
        with open(TARGET, "w", newline="\n") as f:
            f.write(code if code.endswith("\n") else code + "\n")

        result = subprocess.run(
            [PATCH, "-p0", TARGET],
            input=patch,
            text=True,
            capture_output=True,
        )

        # GNU patch semantics:
        # 0 = clean
        # 1 = applied with fuzz/offsets
        # 2 = failed

        if result.returncode == 2:
            return code, False

        with open(TARGET) as f:
            return f.read(), True
    
    except Exception:
        raise

    finally: 
        TARGET.unlink(missing_ok=True)


bad_code=ds_test[0]["bad_code"]+"\n"
good_code=ds_test[0]["good_code"]+"\n"

def normalize(s: str) -> str:
    return s.replace("\r\n", "\n")

patch = create_diff_patch(bad_code, good_code)
print(patch)
fixed_code, _ = apply_diff_patch(bad_code, patch)
print(fixed_code)

print(normalize(fixed_code)== normalize(good_code))

In [ ]:
patchi = create_patch(good_code, good_code)
print(type(patchi))

In [ ]:
import re

HUNK_RE = re.compile(r'^@@ -(\d+),(\d+) \+(\d+),(\d+) @@')

def remove_comments(code: str) -> str:
    """Remove all // commments from code"""
    lines = code.split('\n')
    cleaned_lines = []
    
    for line in lines:
        # Find // and remove everything from that point onward
        comment_index = line.find('//')
        if comment_index != -1:
            line = line[:comment_index]
        
        if line.strip():
            cleaned_lines.append(line)
    
    return '\n'.join(cleaned_lines)


def remove_fileheaders(patch: str) -> str:
    """Remove any +++ or --- file headers from the diff patch"""
    newlines = []
    lines = patch.splitlines()

    for line in lines:
        if not (line.startswith("+++") or line.startswith("---")):
            newlines.append(line)

    return "\n".join(newlines)


def fix_patch_line_counts(patch: str) -> str:
    """Check if the line number counts in each diff header are actually correct, and fix if not"""
    lines = patch.splitlines()
    out = []
    i = 0

    while i < len(lines):
        line = lines[i]
        m = HUNK_RE.match(line)

        if not m:
            out.append(line)
            i += 1
            continue

        old_start, old_cnt, new_start, new_cnt = map(int, m.groups())

        old_actual = 0
        new_actual = 0
        hunk_lines = []
        i += 1

        while i < len(lines):
            l = lines[i]
            if l.startswith('@@ '):
                break
            if l.startswith(' '):
                old_actual += 1
                new_actual += 1
            elif l.startswith('-'):
                old_actual += 1
            elif l.startswith('+'):
                new_actual += 1
            hunk_lines.append(l)
            i += 1

        if old_actual != old_cnt or new_actual != new_cnt:
            line = f"@@ -{old_start},{old_actual} +{new_start},{new_actual} @@"

        out.append(line)
        out.extend(hunk_lines)

    return "\n".join(out)


def validate_patch(patch: str) -> str:
    patch = remove_fileheaders(patch)


In [ ]:
import re
def clear_fileheaders(patch: str):
    newlines= []
    lines = patch.splitlines()
    for line in lines:
        if not (line.startswith("+++") or line.startswith("---")):
            newlines.append(line)
    
    return newlines.join("\n")



def get_line_count(patch: str):
    m = re.match(r"@@ -(\d+),(\d+) \+(\d+),(\d+) @@", patch)
    return m
    


In [ ]:
print(fix_patch_line_counts(patch))

In [ ]:
import difflib, subprocess, tempfile, os
PATCH = r"C:\Program Files\Git\usr\bin\patch.exe"

old_lines = bc.splitlines()
new_lines = gc.splitlines()

diff = difflib.unified_diff(old_lines, new_lines, n=2)

patch = '\n'.join(diff)
patch_text = patch
print(patch_text)






fc = apply_patch(bc, patch_text)

In [ ]:
bad_code = '''package Vehicle_Remix_0085_3e87 {
    port def MotorOutput;
    port def BatteryOutput;
    port def IgnitionCmdPort;
    part def BatteryOutput_Def { port p : BatteryOutput; }
    part def MotorOutput_Def { port p : MotorOutput; }
    part def IgnitionCmdPort_Distractor_Def { port p : IgnitionCmdPort; }
    part def SubSystem_Context {
        part comp_a_9996 : BatteryOutput_Def;
        part comp_b_b5f6 : MotorOutput_Def;
        part comp_distractor_e125 : IgnitionCmdPort_Distractor_Def;
        connect comp_a_9996.p to comp_distractor_e125.p;
    }
}
'''

good_code = '''package Vehicle_Remix_0085_3e87 {
    port def GunOutput;
    port def BatteryOutput;
    port def IgnitionCmdPort;
    part def BatteryOutput_Def { port p : BatteryOutput; }
    part def MotorOutput_Def { port p : MotorOutput; }
    part def IgnitionCmdPort_Distractor_Def { port p : IgnitionCmdPort; }
    part def SubSystem_Context {
        part comp_a_9996 : BatteryOutput_Def;
        part comp_b_b5f6 : MotorOutput_Def;
        part comp_distractor_e125 : IgnitionCmdPort_Distractor_Def;
        connect comp_a_9996.p to comp_gun_e125.p;
    }
}
'''


In [ ]:
import difflib

old_lines = bad_code.splitlines()
new_lines = good_code.splitlines()

diff = difflib.unified_diff(old_lines, new_lines, n=2)

patch = '\n'.join(diff)
patch_text = patch
print(patch_text)
import re

def delete_file_header(s: str) -> str:
    return re.sub(r"\A---\s*\n\s*\n\+\+\+\s*\n\s*\n", "", s, count=1)


patch = delete_file_header(patch)
#print(patch)


In [ ]:
import re

def verify_hunk(hunk: str) -> str:
    lines = hunk.splitlines(keepends=True)
    if not lines:
        return hunk

    header = lines[0]
    body = lines[1:]

    m = re.match(r"@@ -(\d+),(\d+) \+(\d+),(\d+) @@", header)
    if not m:
        return hunk

    before = after = 0
    for line in body:
        if not line:
            continue
        c = line[0]
        if c == '-':
            before += 1
        elif c == '+':
            after += 1
        elif c == ' ':
            before += 1
            after += 1

    old_start, _, new_start, _ = m.groups()
    new_header = f"@@ -{old_start},{before} +{new_start},{after} @@\n"

    if new_header == header:
        return hunk

    return new_header + "".join(body)


def split_hunks(diff: str) -> list[str]:
    hunks = []
    current = []

    for line in diff.splitlines(keepends=True):
        if line.startswith('@@'):
            # close previous hunk
            if current:
                hunks.append("".join(current))
                current = []

            # ensure header ends with newline
            if not line.endswith('\n'):
                line += '\n'

        current.append(line)

    if current:
        hunks.append("".join(current))

    return hunks



In [ ]:
hunks = split_hunks(patch)
print(hunks)




In [ ]:
from pathlib import Path


def save_temp(code: str):
    WRKDIR = Path.cwd()
    with open("t_e_m_p.txt", "w") as f:
        f.write(code)

def delete_temp():
    WRKDIR = Path.cwd()
    (WRKDIR / "t_e_m_p.txt").unlink(missing_ok=True)

def open_temp():
    WRKDIR = Path.cwd()
    with open((WRKDIR / "t_e_m_p.txt"), "r") as f:
        return f.read()


In [ ]:
s = patch_text
s = s.replace('--- ', '--- a/t_e_m_p.txt')
s = s.replace('+++ ', '+++ b/t_e_m_p.txt')
print(s)

In [ ]:
from patch_ng import Patch, fromstring
patch = fromstring(s.encode("utf-8"))
print(patch)

In [ ]:
patch_text = """--- a/t_e_m_p.txt
+++ b/t_e_m_p.txt
@@ -1,4 +1,4 @@

 package Vehicle_Remix_0085_3e87 {
-    port def MotorOutput;
+    port def GunOutput;
     port def BatteryOutput;
     port def IgnitionCmdPort;
"""

In [ ]:
import subprocess
import tempfile
import os

PATCH = r"C:\Program Files\Git\usr\bin\patch.exe"

def apply_patch(code: str, diff: str) -> str:
    with tempfile.NamedTemporaryFile(mode="w+", delete=False) as f:
        f.write(code)
        f.flush()

        result = subprocess.run(
            [PATCH, f.name],
            input=diff,
            text=True,
            capture_output=True,
)

        if result.returncode == 2:
            raise RuntimeError(result.stderr)
    
        print(result)


        f.seek(0)
        result = f.read()

    os.unlink(f.name)
    return result

fc = apply_patch(bc, patch_text)


In [ ]:
print(fc)

In [ ]:
print(gc)

In [ ]:
patch = fromstring("""--- a/t_e_m_p.txt
+++ b/t_e_m_p.txt
@@ -1,4 +1,4 @@

 package Vehicle_Remix_0085_3e87 {
-    port def MotorOutput;
+    port def GunOutput;
     port def BatteryOutput;
     port def IgnitionCmdPort;""".encode("utf-8"))

root = Path.cwd()

result = patch.apply(root=root, strip=1)
print(result)
print(patch.errors)

In [ ]:
patch = fromstring("""--- dev/null
+++ b/t_e_m_p.txt
@@ -1,4 +1,4 @@

 package Vehicle_Remix_0085_3e87 {
-    port def MotorOutput;1q  2mnkj     
                   
+    port def GunOutput;
     port def BatteryOutput;
     port def IgnitionCmdPort;""".encode("utf-8"))
import os
patch.apply(root=os.getcwd(), strip=0).,

In [ ]:
for item in patch.items:
    print("a")
    for hunk in item:
        print("b")
        print(hunk.text)

In [ ]:
b=patch.items[0].hunks[0].text
print(b)

In [ ]:
from os.path import abspath
filename = abspath("t_e_m_p.txt")
print(filename)

In [ ]:
from patch_ng import Patch, fromstring
save_temp(bad_code)
'''
for hunk in hunks:
    patch_text = (
        "--- a/t_e_m_p.txt\n\n"
        "+++ b/t_e_m_p.txt\n\n"
        f"{hunk}"
    )
    print(patch_text)

    patch = fromstring(patch_text.encode("utf-8"))
    root = Path.cwd()

    result = patch.apply(root=root, strip=1)
    print(result)
    print(patch.errors)
'''
patch_text = '''--- a/t_e_m_p.txt  

+++ b/t_e_m_p.txt

@@ -1,4 +1,4 @@

package Vehicle_Remix_0085_3e87 {
-   port def MotorOutput;
+   port def GunOutput;
    port def BatteryOutput;
    port def IgnitionCmdPort;
    
'''


root = Path.cwd()

result = patch.apply(root=root, strip=1)
print(result)
print(patch.errors)

try:
    fixed_code = open_temp()
except:
    raise
finally:
    pass
    #delete_temp() # always delete the temp file

if fixed_code == good_code:
    print("Success")
else:
    print("Code not matching")
    print(fixed_code)

In [ ]:
print(good_code)

In [ ]:
diff = difflib.unified_diff(old_lines, new_lines, n=2)
old_lines = fixed_code.splitlines()
new_lines = good_code.splitlines()
patch = '\n'.join(diff)
print(patch)

In [ ]:
from patch_ng import Patch, fromstring

string = """--- a/newfile
+++ /dev/null
@@ -0,3 +0,0 @@
-New file1
-New file2
-New file3
"""


print(pset)

In [ ]:


#patch = Patch()

pset = fromstring(patch.encode("utf-8"))
#patched_bc = pset.apply(bc)

print(pset)
